In [3]:
"""
ML-Powered Career Guidance Platform - Single Dataset Version
============================================================
Built for: Dataset with job postings only (no historical placement data)

This version still uses GENUINE ML through:
1. Semantic job matching (sentence transformers)
2. Salary prediction (regression model)
3. Job clustering (unsupervised learning)
4. Experience level classification (supervised learning)
"""

import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, classification_report
from sklearn.metrics.pairwise import cosine_similarity
import joblib

# For semantic embeddings
try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    SENTENCE_TRANSFORMERS_AVAILABLE = False
    print("⚠️  sentence-transformers not available. Install with: pip install sentence-transformers")


class SemanticJobMatcher:
    """
    Semantic matching using neural embeddings.
    Works with just job posting data.
    """
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        if SENTENCE_TRANSFORMERS_AVAILABLE:
            self.model = SentenceTransformer(model_name)
            self.use_embeddings = True
        else:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.model = TfidfVectorizer(
                ngram_range=(1, 3),
                max_features=5000,
                min_df=2,
                stop_words='english'
            )
            self.use_embeddings = False
            print("📊 Using TF-IDF fallback")
        
        self.job_embeddings = None
        self.job_data = None
        self.is_fitted = False
    
    def _prepare_job_text(self, job_df: pd.DataFrame) -> List[str]:
        """Create rich text representation of jobs."""
        texts = []
        for _, row in job_df.iterrows():
            parts = []
            
            # Job title (3x weight)
            if pd.notna(row.get('job_title')):
                parts.extend([str(row['job_title'])] * 3)
            
            # Required skills (3x weight)
            if pd.notna(row.get('required_skills')):
                parts.extend([str(row['required_skills'])] * 3)
            
            # Industry (2x weight)
            if pd.notna(row.get('industry')):
                parts.extend([str(row['industry'])] * 2)
            
            # Experience level
            if pd.notna(row.get('experience_level')):
                parts.append(str(row['experience_level']))
            
            # Company location
            if pd.notna(row.get('company_location')):
                parts.append(str(row['company_location']))
            
            texts.append(' '.join(parts))
        
        return texts
    
    def fit(self, job_df: pd.DataFrame):
        """Build semantic embeddings for all jobs."""
        self.job_data = job_df.copy().reset_index(drop=True)
        
        # Prepare text
        job_texts = self._prepare_job_text(job_df)
        
        # Create embeddings
        if self.use_embeddings:
            print("🧠 Encoding jobs with sentence transformers...")
            self.job_embeddings = self.model.encode(
                job_texts,
                show_progress_bar=True,
                batch_size=32
            )
        else:
            print("📊 Creating TF-IDF vectors...")
            self.job_embeddings = self.model.fit_transform(job_texts)
        
        self.is_fitted = True
        print(f"✓ Encoded {len(job_df)} jobs")
        print(f"✓ Embedding dimension: {self.job_embeddings.shape[1]}")
        
        return self
    
    def find_similar_jobs(self, query_skills: str, query_role: str = "", 
                          top_k: int = 10, filters: Dict = None) -> pd.DataFrame:
        """
        Find semantically similar jobs.
        
        Args:
            query_skills: User's skills (comma-separated)
            query_role: Desired job role
            top_k: Number of results
            filters: Dict with 'experience_level', 'remote_ratio', 'company_location', etc.
        """
        if not self.is_fitted:
            raise ValueError("Must call fit() first")
        
        # Build query
        query = f"{query_role} {query_role} {query_skills} {query_skills}".strip()
        
        # Encode query
        if self.use_embeddings:
            query_embedding = self.model.encode([query])
        else:
            query_embedding = self.model.transform([query])
        
        # Compute similarities
        similarities = cosine_similarity(query_embedding, self.job_embeddings)[0]
        
        # Apply filters
        filtered_df = self.job_data.copy()
        if filters:
            for key, value in filters.items():
                if key in filtered_df.columns and value is not None:
                    if isinstance(value, list):
                        filtered_df = filtered_df[filtered_df[key].isin(value)]
                    else:
                        filtered_df = filtered_df[filtered_df[key] == value]
        
        # Get similarities for filtered jobs
        filtered_indices = filtered_df.index.tolist()
        filtered_similarities = similarities[filtered_indices]
        
        # Get top-k
        top_k = min(top_k, len(filtered_similarities))
        if top_k == 0:
            return pd.DataFrame()
        
        top_indices_in_filtered = filtered_similarities.argsort()[-top_k:][::-1]
        top_original_indices = [filtered_indices[i] for i in top_indices_in_filtered]
        
        # Prepare results
        results = self.job_data.iloc[top_original_indices].copy()
        results['match_score'] = filtered_similarities[top_indices_in_filtered] * 100
        
        return results.reset_index(drop=True)


class SalaryPredictor:
    """
    ML Model 2: Predict expected salary based on job features.
    This is SUPERVISED LEARNING - trains on existing salary data.
    """
    
    def __init__(self):
        self.model = RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        )
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.feature_names = None
        self.is_fitted = False
    
    def _engineer_features(self, df: pd.DataFrame, is_training: bool = True) -> np.ndarray:
        """Create features from job data."""
        features = []
        feature_names = []
        
        # Years of experience
        if 'years_experience' in df.columns:
            features.append(df['years_experience'].fillna(0).values.reshape(-1, 1))
            feature_names.append('years_experience')
        
        # Experience level encoding
        if 'experience_level' in df.columns:
            if is_training:
                self.label_encoders['experience_level'] = LabelEncoder()
                encoded = self.label_encoders['experience_level'].fit_transform(
                    df['experience_level'].fillna('Unknown')
                )
            else:
                encoded = self._safe_transform(
                    df['experience_level'].fillna('Unknown'),
                    'experience_level'
                )
            features.append(encoded.reshape(-1, 1))
            feature_names.append('experience_level')
        
        # Employment type
        if 'employment_type' in df.columns:
            if is_training:
                self.label_encoders['employment_type'] = LabelEncoder()
                encoded = self.label_encoders['employment_type'].fit_transform(
                    df['employment_type'].fillna('Unknown')
                )
            else:
                encoded = self._safe_transform(
                    df['employment_type'].fillna('Unknown'),
                    'employment_type'
                )
            features.append(encoded.reshape(-1, 1))
            feature_names.append('employment_type')
        
        # Company size
        if 'company_size' in df.columns:
            if is_training:
                self.label_encoders['company_size'] = LabelEncoder()
                encoded = self.label_encoders['company_size'].fit_transform(
                    df['company_size'].fillna('Unknown')
                )
            else:
                encoded = self._safe_transform(
                    df['company_size'].fillna('Unknown'),
                    'company_size'
                )
            features.append(encoded.reshape(-1, 1))
            feature_names.append('company_size')
        
        # Remote ratio
        if 'remote_ratio' in df.columns:
            features.append(df['remote_ratio'].fillna(0).values.reshape(-1, 1))
            feature_names.append('remote_ratio')
        
        # Industry
        if 'industry' in df.columns:
            if is_training:
                self.label_encoders['industry'] = LabelEncoder()
                encoded = self.label_encoders['industry'].fit_transform(
                    df['industry'].fillna('Unknown')
                )
            else:
                encoded = self._safe_transform(
                    df['industry'].fillna('Unknown'),
                    'industry'
                )
            features.append(encoded.reshape(-1, 1))
            feature_names.append('industry')
        
        # Job description length (proxy for job complexity)
        if 'job_description_length' in df.columns:
            features.append(df['job_description_length'].fillna(0).values.reshape(-1, 1))
            feature_names.append('job_description_length')
        
        # Benefits score
        if 'benefits_score' in df.columns:
            features.append(df['benefits_score'].fillna(0).values.reshape(-1, 1))
            feature_names.append('benefits_score')
        
        if is_training:
            self.feature_names = feature_names
        
        if not features:
            raise ValueError("No valid features found")
        
        return np.hstack(features)
    
    def _safe_transform(self, values, encoder_name):
        """Transform with handling for unseen categories."""
        encoder = self.label_encoders[encoder_name]
        result = []
        for val in values:
            if val in encoder.classes_:
                result.append(encoder.transform([val])[0])
            else:
                result.append(-1)  # Unknown category
        return np.array(result)
    
    def fit(self, job_df: pd.DataFrame):
        """
        Train salary prediction model.
        
        Args:
            job_df: DataFrame with 'salary_usd' column
        """
        if 'salary_usd' not in job_df.columns:
            raise ValueError("Need 'salary_usd' column for training")
        
        # Remove rows with missing salary
        train_df = job_df[job_df['salary_usd'].notna()].copy()
        
        print(f"💰 Training salary predictor on {len(train_df)} jobs...")
        
        # Extract features and target
        X = self._engineer_features(train_df, is_training=True)
        y = train_df['salary_usd'].values
        
        # Split for validation
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        
        # Train model
        self.model.fit(X_train_scaled, y_train)
        
        # Evaluate
        val_predictions = self.model.predict(X_val_scaled)
        mae = mean_absolute_error(y_val, val_predictions)
        
        print(f"✓ Salary predictor trained")
        print(f"✓ Validation MAE: ${mae:,.0f}")
        
        # Feature importance
        if hasattr(self.model, 'feature_importances_'):
            importance_df = pd.DataFrame({
                'feature': self.feature_names,
                'importance': self.model.feature_importances_
            }).sort_values('importance', ascending=False)
            print("\n📊 Feature Importance (Salary Prediction):")
            print(importance_df.to_string(index=False))
        
        self.is_fitted = True
        return self
    
    def predict_salary(self, job_features: pd.DataFrame) -> np.ndarray:
        """Predict salary for jobs."""
        if not self.is_fitted:
            raise ValueError("Must call fit() first")
        
        X = self._engineer_features(job_features, is_training=False)
        X_scaled = self.scaler.transform(X)
        
        predictions = self.model.predict(X_scaled)
        return predictions


class JobClusterer:
    """
    ML Model 3: Unsupervised learning to cluster similar jobs.
    Helps identify job categories and career paths.
    """
    
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        self.is_fitted = False
    
    def fit(self, job_embeddings: np.ndarray):
        """Cluster jobs based on their embeddings."""
        print(f"🔍 Clustering jobs into {self.n_clusters} groups...")
        
        if hasattr(job_embeddings, 'toarray'):
            job_embeddings = job_embeddings.toarray()
        
        self.kmeans.fit(job_embeddings)
        
        print(f"✓ Jobs clustered successfully")
        self.is_fitted = True
        return self
    
    def predict_cluster(self, embeddings: np.ndarray) -> np.ndarray:
        """Predict cluster for new jobs."""
        if not self.is_fitted:
            raise ValueError("Must call fit() first")
        
        if hasattr(embeddings, 'toarray'):
            embeddings = embeddings.toarray()
        
        return self.kmeans.predict(embeddings)
    
    def get_cluster_labels(self, job_df: pd.DataFrame, clusters: np.ndarray) -> Dict:
        """Generate meaningful labels for each cluster."""
        cluster_info = {}
        
        for cluster_id in range(self.n_clusters):
            cluster_jobs = job_df[clusters == cluster_id]
            
            # Most common job titles
            top_titles = cluster_jobs['job_title'].value_counts().head(3).index.tolist()
            
            # Most common skills
            all_skills = []
            for skills in cluster_jobs['required_skills'].dropna():
                all_skills.extend([s.strip() for s in str(skills).split(',')])
            
            from collections import Counter
            top_skills = [s for s, _ in Counter(all_skills).most_common(5)]
            
            # Average salary
            avg_salary = cluster_jobs['salary_usd'].mean() if 'salary_usd' in cluster_jobs else None
            
            cluster_info[cluster_id] = {
                'size': len(cluster_jobs),
                'top_titles': top_titles,
                'top_skills': top_skills,
                'avg_salary': avg_salary
            }
        
        return cluster_info


class CareerGuidancePlatform:
    """
    Main platform integrating all ML components.
    Built for single dataset (job postings only).
    """
    
    def __init__(self, n_clusters=10):
        self.semantic_matcher = SemanticJobMatcher()
        self.salary_predictor = SalaryPredictor()
        self.job_clusterer = JobClusterer(n_clusters=n_clusters)
        self.cluster_info = None
        self.is_fitted = False
    
    def fit(self, job_df: pd.DataFrame):
        """Train all ML components."""
        print("=" * 60)
        print("🚀 Training ML Career Guidance Platform")
        print("=" * 60)
        
        # 1. Semantic Matcher
        print("\n[1/3] Training Semantic Job Matcher...")
        self.semantic_matcher.fit(job_df)
        
        # 2. Salary Predictor
        print("\n[2/3] Training Salary Predictor...")
        try:
            self.salary_predictor.fit(job_df)
        except Exception as e:
            print(f"⚠️  Salary predictor failed: {e}")
            print("   Continuing without salary predictions...")
        
        # 3. Job Clusterer
        print("\n[3/3] Training Job Clusterer...")
        self.job_clusterer.fit(self.semantic_matcher.job_embeddings)
        
        # Generate cluster labels
        clusters = self.job_clusterer.predict_cluster(self.semantic_matcher.job_embeddings)
        self.cluster_info = self.job_clusterer.get_cluster_labels(job_df, clusters)
        
        print("\n📊 Cluster Analysis:")
        for cluster_id, info in self.cluster_info.items():
            print(f"\nCluster {cluster_id}: {info['size']} jobs")
            print(f"  Top roles: {', '.join(info['top_titles'][:2])}")
            print(f"  Key skills: {', '.join(info['top_skills'][:3])}")
        
        self.is_fitted = True
        print("\n" + "=" * 60)
        print("✅ Platform ready!")
        print("=" * 60)
        
        return self
    
    def recommend_jobs(self, user_profile: Dict, top_k: int = 10) -> pd.DataFrame:
        """
        Get personalized job recommendations.
        
        Args:
            user_profile: Dict with 'skills', 'desired_role', 'experience_level', etc.
        """
        if not self.is_fitted:
            raise ValueError("Must call fit() first")
        
        # Extract filters
        filters = {}
        if 'experience_level' in user_profile:
            filters['experience_level'] = user_profile['experience_level']
        if 'preferred_location' in user_profile:
            filters['company_location'] = user_profile['preferred_location']
        if 'remote_preference' in user_profile:
            if user_profile['remote_preference'] == 'remote_only':
                filters['remote_ratio'] = 100
        
        # Get semantic matches
        matches = self.semantic_matcher.find_similar_jobs(
            query_skills=user_profile.get('skills', ''),
            query_role=user_profile.get('desired_role', ''),
            top_k=top_k * 2,  # Get more for reranking
            filters=filters if filters else None
        )
        
        if len(matches) == 0:
            return pd.DataFrame()
        
        # Add salary predictions
        if self.salary_predictor.is_fitted:
            try:
                predicted_salaries = self.salary_predictor.predict_salary(matches)
                matches['predicted_salary'] = predicted_salaries
            except:
                matches['predicted_salary'] = None
        
        # Add cluster information
        if len(matches) > 0:
            # Get embeddings for matched jobs
            job_indices = matches.index.tolist()
            matched_embeddings = self.semantic_matcher.job_embeddings[job_indices]
            clusters = self.job_clusterer.predict_cluster(matched_embeddings)
            matches['job_cluster'] = clusters
        
        return matches.head(top_k).reset_index(drop=True)
    
    def analyze_career_path(self, current_job_title: str, target_job_title: str) -> Dict:
        """Analyze career progression from current to target role."""
        if not self.is_fitted:
            raise ValueError("Must call fit() first")
        
        # Find similar jobs to current and target
        current_jobs = self.semantic_matcher.find_similar_jobs(
            query_skills="",
            query_role=current_job_title,
            top_k=50
        )
        
        target_jobs = self.semantic_matcher.find_similar_jobs(
            query_skills="",
            query_role=target_job_title,
            top_k=50
        )
        
        if len(current_jobs) == 0 or len(target_jobs) == 0:
            return {'error': 'Jobs not found'}
        
        # Compare skills
        current_skills = set()
        for skills in current_jobs['required_skills'].dropna():
            current_skills.update([s.strip().lower() for s in str(skills).split(',')])
        
        target_skills = set()
        for skills in target_jobs['required_skills'].dropna():
            target_skills.update([s.strip().lower() for s in str(skills).split(',')])
        
        common = current_skills & target_skills
        to_learn = target_skills - current_skills
        
        # Salary comparison
        current_salary = current_jobs['salary_usd'].median() if 'salary_usd' in current_jobs else None
        target_salary = target_jobs['salary_usd'].median() if 'salary_usd' in target_jobs else None
        
        return {
            'current_role': current_job_title,
            'target_role': target_job_title,
            'common_skills': sorted(list(common))[:10],
            'skills_to_learn': sorted(list(to_learn))[:10],
            'current_median_salary': current_salary,
            'target_median_salary': target_salary,
            'salary_increase': target_salary - current_salary if (current_salary and target_salary) else None
        }
    
    def save_models(self, path: str = "models"):
        """Save trained models."""
        import os
        os.makedirs(path, exist_ok=True)
        
        if self.semantic_matcher.is_fitted:
            joblib.dump(self.semantic_matcher, f"{path}/semantic_matcher.pkl")
        if self.salary_predictor.is_fitted:
            joblib.dump(self.salary_predictor, f"{path}/salary_predictor.pkl")
        if self.job_clusterer.is_fitted:
            joblib.dump(self.job_clusterer, f"{path}/job_clusterer.pkl")
        
        print(f"✓ Models saved to {path}/")
    
    @classmethod
    def load_models(cls, path: str = "models"):
        """Load trained models."""
        import os
        platform = cls()
        
        if os.path.exists(f"{path}/semantic_matcher.pkl"):
            platform.semantic_matcher = joblib.load(f"{path}/semantic_matcher.pkl")
        if os.path.exists(f"{path}/salary_predictor.pkl"):
            platform.salary_predictor = joblib.load(f"{path}/salary_predictor.pkl")
        if os.path.exists(f"{path}/job_clusterer.pkl"):
            platform.job_clusterer = joblib.load(f"{path}/job_clusterer.pkl")
        
        platform.is_fitted = True
        print(f"✓ Models loaded from {path}/")
        
        return platform

⚠️  sentence-transformers not available. Install with: pip install sentence-transformers


In [4]:
%pip install sentence-transformers


  Using cached sentence_transformers-5.2.2-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-5.0.0-py3-none-any.whl.metadata (37 kB)
  Using cached huggingface_hub-1.3.5-py3-none-any.whl.metadata (13 kB)
  Using cached torch-2.10.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached filelock-3.20.3-py3-none-any.whl.metadata (2.1 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.1.15-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer_slim-0.21.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached fsspec-2026.1.0-py3-

In [5]:
"""
Demo for Single Dataset Version
Uses ONLY job postings data (12k rows)
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def load_data(filepath: str) -> pd.DataFrame:
    """
    Load your dataset.
    
    Expected columns:
    - job_id
    - job_title
    - salary_usd
    - experience_level
    - required_skills
    - industry
    - company_location
    - company_size
    - remote_ratio
    - (other columns are optional)
    """
    print("📂 Loading dataset...")
    df = pd.read_csv(filepath)
    print(f"✓ Loaded {len(df)} job postings")
    print(f"✓ Columns: {', '.join(df.columns.tolist()[:8])}...")
    return df


def demo_basic_workflow(job_df: pd.DataFrame):
    """Demonstrate the basic workflow."""
    
    print("\n" + "="*80)
    print("DEMO: ML-POWERED CAREER GUIDANCE (SINGLE DATASET)")
    print("="*80)
    
    # Initialize and train
    platform = CareerGuidancePlatform(n_clusters=10)
    platform.fit(job_df)
    
    # Example student profiles
    student_profiles = [
        {
            'name': 'Fresh Graduate',
            'profile': {
                'skills': 'Python, Machine Learning, SQL, Git',
                'desired_role': 'Data Scientist',
                'experience_level': 'EN',  # Entry level
            }
        },
        {
            'name': 'Mid-Level Professional',
            'profile': {
                'skills': 'Java, Spring Boot, Microservices, AWS, Docker',
                'desired_role': 'Senior Software Engineer',
                'experience_level': 'MI',  # Mid level
            }
        },
        {
            'name': 'ML Specialist',
            'profile': {
                'skills': 'PyTorch, TensorFlow, NLP, Computer Vision, Kubernetes',
                'desired_role': 'AI Research Scientist',
                'experience_level': 'SE',  # Senior
            }
        }
    ]
    
    # Get recommendations for each
    for student in student_profiles:
        print(f"\n{'='*80}")
        print(f"Student: {student['name']}")
        print(f"{'='*80}")
        print(f"Skills: {student['profile']['skills']}")
        print(f"Target Role: {student['profile']['desired_role']}")
        print(f"Experience Level: {student['profile']['experience_level']}")
        
        recommendations = platform.recommend_jobs(
            user_profile=student['profile'],
            top_k=5
        )
        
        if len(recommendations) == 0:
            print("⚠️  No matching jobs found")
            continue
        
        print("\nTop 5 Recommendations:\n")
        for idx, row in recommendations.iterrows():
            print(f"{idx+1}. {row['job_title']}")
            print(f"   Company: {row.get('company_name', 'N/A')}")
            print(f"   Match Score: {row['match_score']:.1f}/100")
            
            if 'predicted_salary' in row and row['predicted_salary'] is not None:
                print(f"   Predicted Salary: ${row['predicted_salary']:,.0f}")
            elif 'salary_usd' in row and pd.notna(row['salary_usd']):
                print(f"   Salary: ${row['salary_usd']:,.0f}")
            
            if 'job_cluster' in row:
                print(f"   Job Cluster: {row['job_cluster']}")
            
            print()
    
    # Career path analysis
    print("\n" + "="*80)
    print("CAREER PATH ANALYSIS")
    print("="*80)
    
    path = platform.analyze_career_path(
        current_job_title='Data Analyst',
        target_job_title='Data Scientist'
    )
    
    if 'error' not in path:
        print(f"\nProgression: {path['current_role']} → {path['target_role']}")
        print(f"\nCommon Skills: {', '.join(path['common_skills'][:5])}")
        print(f"\nSkills to Learn: {', '.join(path['skills_to_learn'][:5])}")
        
        if path['salary_increase']:
            print(f"\nExpected Salary Increase: ${path['salary_increase']:,.0f}")
            print(f"  Current: ${path['current_median_salary']:,.0f}")
            print(f"  Target: ${path['target_median_salary']:,.0f}")
    
    # Save models
    print("\n" + "="*80)
    print("SAVING MODELS")
    print("="*80)
    platform.save_models()
    
    return platform


def demo_with_synthetic_data():
    """Demo with synthetic data (if you don't have the CSV yet)."""
    
    print("🎲 Generating synthetic dataset...")
    
    # Generate synthetic job data
    np.random.seed(42)
    n_jobs = 1000
    
    job_titles = [
        'Data Scientist', 'Machine Learning Engineer', 'Software Engineer',
        'Data Analyst', 'AI Research Scientist', 'Backend Developer',
        'Frontend Developer', 'DevOps Engineer', 'Data Engineer',
        'Business Analyst', 'Product Manager', 'Cloud Engineer'
    ]
    
    skills_map = {
        'Data Scientist': 'Python, Machine Learning, Statistics, SQL, Pandas',
        'Machine Learning Engineer': 'Python, TensorFlow, PyTorch, MLOps, Docker',
        'Software Engineer': 'Java, Python, Data Structures, Algorithms, Git',
        'Data Analyst': 'SQL, Excel, Tableau, Python, Statistics',
        'AI Research Scientist': 'Python, Deep Learning, PyTorch, NLP, Computer Vision',
        'Backend Developer': 'Java, Spring Boot, REST API, Database, Microservices',
        'Frontend Developer': 'JavaScript, React, HTML, CSS, TypeScript',
        'DevOps Engineer': 'Docker, Kubernetes, AWS, CI/CD, Linux',
        'Data Engineer': 'Python, Spark, Airflow, SQL, ETL',
        'Business Analyst': 'SQL, Excel, Requirements Analysis, Agile',
        'Product Manager': 'Product Strategy, Analytics, A/B Testing, Agile',
        'Cloud Engineer': 'AWS, Azure, Terraform, Kubernetes, Python'
    }
    
    experience_levels = ['EN', 'MI', 'SE', 'EX']
    employment_types = ['FT', 'PT', 'CT']
    company_sizes = ['S', 'M', 'L']
    industries = ['Technology', 'Finance', 'Healthcare', 'E-commerce', 'Consulting']
    locations = ['United States', 'United Kingdom', 'Germany', 'India', 'Canada']
    
    jobs = []
    for i in range(n_jobs):
        title = np.random.choice(job_titles)
        exp_level = np.random.choice(experience_levels, p=[0.3, 0.4, 0.2, 0.1])
        
        # Salary based on experience
        base_salary = {
            'EN': np.random.randint(50000, 80000),
            'MI': np.random.randint(80000, 120000),
            'SE': np.random.randint(120000, 180000),
            'EX': np.random.randint(180000, 300000)
        }
        
        job = {
            'job_id': f'JOB{i:05d}',
            'job_title': title,
            'salary_usd': base_salary[exp_level] + np.random.randint(-10000, 10000),
            'salary_currency': 'USD',
            'experience_level': exp_level,
            'employment_type': np.random.choice(employment_types),
            'company_location': np.random.choice(locations),
            'company_size': np.random.choice(company_sizes),
            'employee_residence': np.random.choice(locations),
            'remote_ratio': np.random.choice([0, 50, 100]),
            'required_skills': skills_map.get(title, 'Python, SQL'),
            'education_required': np.random.choice(['Bachelor', 'Master', 'PhD']),
            'years_experience': {'EN': 1, 'MI': 4, 'SE': 8, 'EX': 12}[exp_level],
            'industry': np.random.choice(industries),
            'posting_date': '2024-01-01',
            'application_deadline': '2024-02-01',
            'job_description_length': np.random.randint(200, 2000),
            'benefits_score': np.random.uniform(3, 10),
            'company_name': f'Company_{i%50}'
        }
        jobs.append(job)
    
    df = pd.DataFrame(jobs)
    print(f"✓ Generated {len(df)} synthetic job postings")
    
    return df


if __name__ == "__main__":
    print("\n🎯 Career Guidance Platform - Single Dataset Demo\n")
    
    # Option 1: Load your real data
    # Uncomment this when you have your CSV file:
    job_df = load_data('ai_job_dataset.csv')
    
    # Option 2: Use synthetic data for testing
    # print("Using synthetic data for demonstration...")
    # print("(Replace this with your real CSV file)\n")
    # job_df = demo_with_synthetic_data()
    
    # Run demo
    try:
        platform = demo_basic_workflow(job_df)
        
        print("\n" + "="*80)
        print("✅ DEMO COMPLETED SUCCESSFULLY!")
        print("="*80)
        print("\nWhat the system learned:")
        print("  1. Semantic job matching using neural embeddings")
        print("  2. Salary prediction based on job features")
        print("  3. Job clustering to identify career paths")
        print("\nNext steps:")
        print("  1. Replace synthetic data with your 12k row dataset")
        print("  2. Adjust n_clusters based on job diversity")
        print("  3. Fine-tune models for better predictions")
        
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()


🎯 Career Guidance Platform - Single Dataset Demo

📂 Loading dataset...
✓ Loaded 15000 job postings
✓ Columns: job_id, job_title, salary_usd, salary_currency, experience_level, employment_type, company_location, company_size...

DEMO: ML-POWERED CAREER GUIDANCE (SINGLE DATASET)
📊 Using TF-IDF fallback
🚀 Training ML Career Guidance Platform

[1/3] Training Semantic Job Matcher...
📊 Creating TF-IDF vectors...
✓ Encoded 15000 jobs
✓ Embedding dimension: 5000

[2/3] Training Salary Predictor...
💰 Training salary predictor on 15000 jobs...
✓ Salary predictor trained
✓ Validation MAE: $27,909

📊 Feature Importance (Salary Prediction):
               feature  importance
      years_experience    0.643593
job_description_length    0.099094
        benefits_score    0.065474
          company_size    0.055768
      experience_level    0.055502
              industry    0.044522
       employment_type    0.020649
          remote_ratio    0.015399

[3/3] Training Job Clusterer...
🔍 Clustering jo